# unlocr on GPU (full DeepSeek-OCR model via vLLM)

Run the **full, unquantized** DeepSeek-OCR model on a Colab GPU and OCR PDFs with the `unlocr` CLI.
Everything runs inside this Colab VM: vLLM serves the model on `localhost:8000`, and `unlocr --gpu` connects to it.

**Before you start:** set a GPU runtime via **Runtime > Change runtime type > T4 / L4 / A100** (16 GB+ VRAM).
The model weights (~6.7 GB) download on first serve.

See the [unlocr README](https://github.com/whit3rabbit/unlocr#run-the-full-model-on-gpu-vllm) for the local (non-Colab) version.

In [ ]:
# 0. Confirm a GPU is attached (fails loudly if you forgot to pick a GPU runtime).
!nvidia-smi || (echo 'No GPU. Runtime > Change runtime type > GPU (T4/L4/A100).' && false)

In [ ]:
# 1. poppler-utils provides pdftoppm, which unlocr uses to rasterize PDF pages.
!apt-get -qq update && apt-get -qq install -y poppler-utils
!pdftoppm -v

In [ ]:
# 2. Install vLLM. Stock 0.11 does not yet support DeepSeek-OCR; use the nightly build.
#    This takes a few minutes.
!pip install -q -U vllm --pre --extra-index-url https://wheels.vllm.ai/nightly

In [ ]:
# 3. Get the unlocr Linux x86_64 binary from the latest GitHub release.
#    If no matching release asset is found, run the build-from-source cell below instead.
import json, os, tarfile, urllib.request

UNLOCR = "/content/unlocr"
API = "https://api.github.com/repos/whit3rabbit/unlocr/releases/latest"
rel = json.load(urllib.request.urlopen(API))
asset = next(
    (a for a in rel.get("assets", [])
     if ("linux" in a["name"].lower())
     and ("x86_64" in a["name"].lower() or "x64" in a["name"].lower() or "amd64" in a["name"].lower())
     and a["name"].endswith((".tar.gz", ".tgz"))),
    None,
)
if asset is None:
    raise SystemExit(
        "No Linux x86_64 release tarball found for unlocr. "
        "Run the 'Build from source' cell below instead."
    )
print("Downloading", asset["name"])
urllib.request.urlretrieve(asset["browser_download_url"], "/content/unlocr.tar.gz")
with tarfile.open("/content/unlocr.tar.gz") as t:
    member = next(m for m in t.getmembers() if os.path.basename(m.name) == "unlocr")
    member.name = os.path.basename(member.name)
    t.extract(member, "/content")
os.chmod(UNLOCR, 0o755)
!{UNLOCR} --version

In [ ]:
# 3b. Build from source (fallback). Only run this if the release-download cell failed.
#     Compiles the CLI with cargo; takes a few minutes.
# !curl -sSf https://sh.rustup.rs | sh -s -- -y
# !. "$HOME/.cargo/env" && cargo install --git https://github.com/whit3rabbit/unlocr unlocr --root /content/.cargo
# import os; UNLOCR = "/content/.cargo/bin/unlocr"
# !{UNLOCR} --version

In [ ]:
# 4. Start vLLM serving DeepSeek-OCR on localhost:8000, in the background.
#    First run downloads ~6.7 GB of weights, so readiness can take several minutes.
import subprocess, time, urllib.request

LOG = "/content/vllm.log"
cmd = [
    "vllm", "serve", "deepseek-ai/DeepSeek-OCR",
    "--no-enable-prefix-caching",
    "--mm-processor-cache-gb", "0",
    "--logits-processors", "vllm.model_executor.models.deepseek_ocr:NGramPerReqLogitsProcessor",
]
logf = open(LOG, "w")
proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)

def ready(timeout_s=1800):
    start = time.time()
    while time.time() - start < timeout_s:
        if proc.poll() is not None:
            print(open(LOG).read()[-3000:])
            raise SystemExit(f"vLLM exited early (code {proc.returncode}); see log above.")
        try:
            with urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5) as r:
                if r.status == 200:
                    print("vLLM ready on http://localhost:8000")
                    return
        except Exception:
            pass
        time.sleep(5)
    raise SystemExit("vLLM did not become ready in time; check /content/vllm.log.")

ready()

In [ ]:
# 5. Upload a PDF to OCR.
from google.colab import files
uploaded = files.upload()
PDF = next(iter(uploaded))
print("Uploaded:", PDF)

In [ ]:
# 6. OCR the PDF against the local vLLM server. --gpu = --endpoint http://localhost:8000
#    --endpoint-model deepseek-ai/DeepSeek-OCR.
!{UNLOCR} "{PDF}" --gpu -o /content/out.md
print(open("/content/out.md").read()[:2000])

In [ ]:
# 7. Download the markdown result.
from google.colab import files
files.download("/content/out.md")